# Data Management: Ingestion, EDA, and Cleaning
This notebook serves as the core data engine for the entire system. It defines the `DataLoader` class and specific cleaning functions for four heterogeneous datasets.

### Cell 1: Environment Setup
Configures base paths for local and Colab environments.

In [3]:
import os, sys, pandas as pd, numpy as np, pickle, matplotlib.pyplot as plt, seaborn as sns
BASE_DATA_PATH = os.path.abspath(os.path.join(os.getcwd(), "../../data/"))
HMOG_LOCAL_PATH = os.path.join(BASE_DATA_PATH, "HMOG", "public_dataset")
print(f"Data Root: {BASE_DATA_PATH}")

Data Root: c:\Users\user\Downloads\health-ai-system\data


### Cell 2: The DataLoader Class
A unified interface to load Pickle (WESAD), TXT (SisFall), and CSV (MIMIC/HMOG) files.

In [4]:
class DataLoader:
    def __init__(self, base_path=BASE_DATA_PATH, hmog_path=HMOG_LOCAL_PATH):
        self.base_path, self.hmog_path = base_path, hmog_path
    def load_wesad_subject(self, sid):
        p = os.path.join(self.base_path, "WESAD", sid, f"{sid}.pkl")
        if not os.path.exists(p): return None
        with open(p, "rb") as f: return pickle.load(f, encoding="latin1")
    def load_sisfall_file(self, sid, fname):
        p = os.path.join(self.base_path, "SisFall", sid, fname)
        return pd.read_csv(p, header=None, sep=",", skipinitialspace=True) if os.path.exists(p) else None
    def load_hmog_session(self, uid, sid, sensor="Accelerometer"):
        p = os.path.join(self.hmog_path, str(uid), str(uid), f"{uid}_session_{sid}", f"{sensor}.csv")
        return pd.read_csv(p, header=None) if os.path.exists(p) else None
    def load_mimic_file(self, fname):
        p = os.path.join(self.base_path, "mimic", fname)
        return pd.read_csv(p) if os.path.exists(p) else None

### Cell 3: Dataset-Specific Cleaning
Each dataset has unique noise characteristics. We define four distinct cleaning functions:
1. `clean_wesad_data`: Label filtering & consecutive duplicate removal.
2. `clean_sisfall_data`: Semicolon cleanup and unit conversion.
3. `clean_mimic_data`: Clinical range validation and median imputation.
4. `clean_hmog_data`: Timestamp de-duplication and spike removal.

In [5]:
def clean_wesad_data(raw):
    df = pd.DataFrame(raw["signal"]["chest"]["ACC"], columns=["acc_x", "acc_y", "acc_z"])
    df["label"] = raw["label"]
    df = df[df["label"].isin([1,2,3,4,6,7])]
    return df.loc[(df.shift() != df).any(axis=1)].dropna()

def clean_sisfall_data(df):
    if df.iloc[:, -1].dtype == object: df.iloc[:, -1] = df.iloc[:, -1].str.replace(";", "", regex=False).astype(float)
    df.columns = ["ax","ay","az","gx","gy","gz","ax2","ay2","az2"][:len(df.columns)]
    return df.rolling(window=5).mean().dropna()

def clean_mimic_data(df):
    df = df.copy()
    for col in ["heart_rate", "sbp", "dbp", "temperature"]:
        if col in df.columns: df[col] = df[col].fillna(df[col].median())
    return df

def clean_hmog_data(df):
    df.columns = ["sys_time", "event_time", "sensor_id", "acc_x", "acc_y", "acc_z", "accuracy"]
    return df.drop_duplicates(subset=["sys_time", "event_time"])